
# GVH Diagonal Cubic 0.3.2.7.3.7.2 — Full Coupled Kinetic Matrix Assembly and Generic-Branch Inversion

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.7.3.7.2

## But

`0.3.2.7.3.7.1` a identifié le verrou :

\[
(\pi^{ij},p_s,p_v^i)
\leftrightarrow
(K_{ij},\mathcal D_\perp s,\mathcal D_\perp v_i)
\]

n'était pas encore inversé sous forme générale.

Ce notebook construit un **bloc cinétique couplé explicite de branche générique**, mesure son rang, calcule son déterminant et fournit l'inversion symbolique du sous-secteur couplé.

### Discipline

Le notebook ne prétend pas encore dériver les densités complètes
\(\mathcal C_\perp^{(u)}\) et \(\mathcal C_i^{(u)}\).
Il prépare l'inversion nécessaire à cette étape.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

import sympy as sp, json
from pathlib import Path
print("GVH 0.3.2.7.3.7.2")
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.2
SymPy: 1.14.0



# 1. Variables cinétiques

On organise le vecteur de vitesses comme :

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},
S,V_1,V_2,V_3),
\]

où

\[
S\equiv\mathcal D_\perp s,
\qquad
V_i\equiv\mathcal D_\perp v_i.
\]

Le bloc métrique Einstein-Hilbert est modélisé par la supermétrique ADM standard, tandis que le secteur directionnel ajoute les couplages déjà identifiés dans 0.3.2.7.3.4.


In [2]:

# Generic symbolic parameters
alpha, beta = sp.symbols("alpha beta", nonzero=True, real=True)
c14, ct = sp.symbols("c14 c_time", nonzero=True, real=True)

# Coupling parameters between metric velocities and directional velocities
g1,g2,g3 = sp.symbols("g1 g2 g3", real=True)

# velocity basis
K11,K22,K33,K12,K13,K23,S,V1,V2,V3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S V1 V2 V3", real=True
)
vel = sp.Matrix([K11,K22,K33,K12,K13,K23,S,V1,V2,V3])

# Metric kinetic block: DeWitt-like diagonal proxy in orthonormal local frame.
# alpha controls traceless directions, beta controls trace combination.
Gm = sp.diag(alpha,alpha,alpha,2*alpha,2*alpha,2*alpha)
trace_vec = sp.Matrix([1,1,1,0,0,0])
Gm = Gm + beta*(trace_vec*trace_vec.T)

# Directional block
Gd = sp.diag(-2*ct, 2*c14, 2*c14, 2*c14)

# Coupling matrix: generic local couplings preserving a simple tensorial channel split
Coup = sp.Matrix([
    [g1, g2, 0, 0],
    [g1, 0, g2, 0],
    [g1, 0, 0, g2],
    [0,  g3,0,0],
    [0,  0,g3,0],
    [0,  0,0,g3],
])

Q = sp.Matrix.vstack(
    sp.Matrix.hstack(Gm, Coup),
    sp.Matrix.hstack(Coup.T, Gd)
)

print("Q shape =", Q.shape)
assert Q.shape==(10,10)


Q shape = (10, 10)



# 2. Hessien cinétique explicite

On définit le Lagrangien quadratique local :

\[
\mathcal L_{\rm kin}
=
\frac12 V^TQV.
\]

Les moments sont

\[
P_A=\frac{\partial\mathcal L_{\rm kin}}{\partial V^A}=Q_{AB}V^B.
\]

Sur une branche générique, l'inversion existe si

\[
\det Q\neq0.
\]


In [3]:

Lkin = sp.expand(sp.Rational(1,2)*(vel.T*Q*vel)[0])
P = sp.Matrix([sp.diff(Lkin,x) for x in vel])
assert sp.simplify(P-Q*vel)==sp.zeros(10,1)
print("PASS — P = Q V")


PASS — P = Q V



# 3. Réduction par complément de Schur

Au lieu d'inverser naïvement le \(10\times10\), on exploite :

\[
Q=
\begin{pmatrix}
G_m & C\\
C^T & G_d
\end{pmatrix}.
\]

Si \(G_m\) est inversible, le déterminant factorise :

\[
\det Q
=
\det G_m\,
\det\left(G_d-C^TG_m^{-1}C\right).
\]

Le second facteur est le complément de Schur directionnel.


In [4]:

detGm = sp.factor(Gm.det())
print("det(Gm) =", detGm)

Gm_inv = sp.simplify(Gm.inv())
Schur = sp.simplify(Gd - Coup.T*Gm_inv*Coup)

print("Schur shape =", Schur.shape)
sp.pprint(Schur)


det(Gm) = 8*alpha**5*(alpha + 3*beta)
Schur shape = (4, 4)
⎡                         2                                                    ↪
⎢-2⋅cₜᵢₘₑ⋅(α + 3⋅β) - 3⋅g₁                         -g₁⋅g₂                      ↪
⎢──────────────────────────                        ───────                     ↪
⎢         α + 3⋅β                                  α + 3⋅β                     ↪
⎢                                                                              ↪
⎢                                                    2               2         ↪
⎢         -g₁⋅g₂             4⋅α⋅c₁₄⋅(α + 3⋅β) - 2⋅g₂ ⋅(α + 2⋅β) - g₃ ⋅(α + 3⋅ ↪
⎢         ───────            ───────────────────────────────────────────────── ↪
⎢         α + 3⋅β                               2⋅α⋅(α + 3⋅β)                  ↪
⎢                                                                              ↪
⎢                                                       2                      ↪
⎢         -g₁⋅g₂                                  


# 4. Déterminant et rang générique

Le gate principal est :

\[
\boxed{
\det Q=\det G_m\det S_{\rm Schur}
}
\]

avec \(S_{\rm Schur}\) le complément de Schur.

Le notebook enregistre les facteurs de dégénérescence sans les interpréter prématurément comme contraintes physiques universelles.


In [5]:

detSchur = sp.factor(Schur.det())
detQ_factored = sp.factor(detGm*detSchur)

print("det(Schur) =")
sp.pprint(detSchur)
print("\ndet(Q) factored =")
sp.pprint(detQ_factored)

# numerical generic branch witness
subs_generic = {
    alpha:2, beta:1,
    c14:3, ct:5,
    g1:sp.Rational(1,3),
    g2:sp.Rational(1,4),
    g3:sp.Rational(1,5),
}
Qnum = Q.subs(subs_generic)
assert Qnum.det()!=0
assert Qnum.rank()==10
print("\nGeneric witness rank(Q) =", Qnum.rank())


det(Schur) =
                        2                                                      ↪
 ⎛              2     2⎞  ⎛   2                                           2    ↪
-⎝4⋅α⋅c₁₄ - 2⋅g₂  - g₃ ⎠ ⋅⎝8⋅α ⋅c₁₄⋅cₜᵢₘₑ + 24⋅α⋅β⋅c₁₄⋅cₜᵢₘₑ + 12⋅α⋅c₁₄⋅g₁  -  ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                3              ↪
                                                             8⋅α ⋅(α + 3⋅β)    ↪

↪                                                            
↪             2               2               2       2   2⎞ 
↪ 4⋅α⋅cₜᵢₘₑ⋅g₂  - 2⋅α⋅cₜᵢₘₑ⋅g₃  - 6⋅β⋅cₜᵢₘₑ⋅g₃  - 3⋅g₁ ⋅g₃ ⎠ 
↪ ───────────────────────────────────────────────────────────
↪                                                            
↪                                                            

det(Q) factored =
                           2                                                   ↪
  2 ⎛              2     2⎞ 


# 5. Inversion générique du bloc

Lorsque \(G_m\) et le complément de Schur sont inversibles :

\[
Q^{-1}
=
\begin{pmatrix}
G_m^{-1}+G_m^{-1}CS^{-1}C^TG_m^{-1}
&
-G_m^{-1}CS^{-1}
\\
-S^{-1}C^TG_m^{-1}
&
S^{-1}
\end{pmatrix}.
\]

Cette forme est exacte au niveau du modèle cinétique assemblé ici.


In [6]:

Schur_inv = sp.simplify(Schur.inv())

Qinv_block = sp.Matrix.vstack(
    sp.Matrix.hstack(
        Gm_inv + Gm_inv*Coup*Schur_inv*Coup.T*Gm_inv,
        -Gm_inv*Coup*Schur_inv
    ),
    sp.Matrix.hstack(
        -Schur_inv*Coup.T*Gm_inv,
        Schur_inv
    )
)

# validate on generic numerical witness
Qinv_num = sp.simplify(Qinv_block.subs(subs_generic))
assert sp.simplify(Qnum*Qinv_num-sp.eye(10))==sp.zeros(10)
print("PASS — generic block inverse validated on witness.")


PASS — generic block inverse validated on witness.



# 6. Carte moment-vitesse

Sur la branche générique :

\[
\boxed{
V^A=(Q^{-1})^{AB}P_B.
}
\]

Cela fournit formellement :

\[
K_{ij}=K_{ij}(P),
\qquad
\mathcal D_\perp s=\mathcal D_\perp s(P),
\qquad
\mathcal D_\perp v_i=\mathcal D_\perp v_i(P).
\]

La prochaine étape devra remplacer les symboles de ce modèle local par les coefficients tensoriels exacts issus du secteur ADM complet.


In [7]:

Psyms = sp.Matrix(sp.symbols("P0:10", real=True))
Vsol = sp.simplify(Qinv_block*Psyms)

print("Number of inverted velocity components =", len(Vsol))
assert len(Vsol)==10


Number of inverted velocity components = 10



# 7. Gates scientifiques

Ce notebook ferme l'existence et la méthode d'inversion sur un bloc générique explicite.

Il ne ferme pas encore :

- l'identification exacte de tous les coefficients \(g_1,g_2,g_3,\alpha,\beta\) avec le Lagrangien GVH ADM complet ;
- les densités canoniques finales \(\mathcal C_\perp^{(u)}\), \(\mathcal C_i^{(u)}\) ;
- l'algèbre hypersurface.

Donc le statut doit rester `PARTIAL PASS`.


In [8]:

GATES = {
    "kinetic_basis_registered":True,
    "explicit_10x10_Q_assembled":True,
    "momenta_equal_QV":True,
    "Schur_complement_derived":True,
    "generic_nonzero_det_witness":True,
    "generic_rank_10_witness":True,
    "generic_block_inverse_constructed":True,
    "generic_inverse_validated":True,

    "exact_GVH_tensor_coefficients_inserted":False,
    "exact_full_field_Q_inverted":False,
    "explicit_Cperp_vector_density":False,
    "explicit_Ci_vector_density":False,
    "hypersurface_algebra_closed":False,
}

for k,v in GATES.items():
    print(k,":",v)

PARTIAL_PASS = all(list(GATES.values())[:8])
FULL_PASS = all(GATES.values())

assert PARTIAL_PASS is True
assert FULL_PASS is False

FINAL_STATUS = (
    "PARTIAL-PASS-GENERIC-COUPLED-KINETIC-MATRIX-ASSEMBLED-AND-INVERTED_"
    "BLOCKED-EXACT-GVH-TENSOR-COEFFICIENT-MAP-AND-FULL-FIELD-CONSTRAINT-DENSITIES"
)

DISPERSION_READY=False
print("\nFINAL STATUS:", FINAL_STATUS)
print("DISPERSION_READY =", DISPERSION_READY)


kinetic_basis_registered : True
explicit_10x10_Q_assembled : True
momenta_equal_QV : True
Schur_complement_derived : True
generic_nonzero_det_witness : True
generic_rank_10_witness : True
generic_block_inverse_constructed : True
generic_inverse_validated : True
exact_GVH_tensor_coefficients_inserted : False
exact_full_field_Q_inverted : False
explicit_Cperp_vector_density : False
explicit_Ci_vector_density : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-GENERIC-COUPLED-KINETIC-MATRIX-ASSEMBLED-AND-INVERTED_BLOCKED-EXACT-GVH-TENSOR-COEFFICIENT-MAP-AND-FULL-FIELD-CONSTRAINT-DENSITIES
DISPERSION_READY = False



# 8. Interprétation stricte

Ce notebook montre qu'un bloc cinétique couplé de la forme requise peut être inversé sur une branche générique non dégénérée.

Il **ne démontre pas encore** que le \(Q_{AB}\) exact de GVH est identique à cette paramétrisation locale.

Le prochain notebook doit donc effectuer la substitution exacte des coefficients tensoriels issus de `0.3.2.7.3.4`, puis reconstruire le \(Q_{AB}\) full-field sans paramètres proxies.



# 9. Étape suivante

## 0.3.2.7.3.7.2.1 — Exact GVH Tensor-Coefficient Lift into the Coupled Kinetic Matrix

Objectifs :

1. remplacer
   \[
   \alpha,\beta,g_1,g_2,g_3
   \]
   par les expressions exactes en
   \[
   h_{ij},s,v_i,c_1,c_2,c_3,c_4;
   \]

2. reconstruire le Hessien cinétique exact ;

3. recalculer son rang et son déterminant ;

4. tenter l'inversion symbolique exacte sur la branche générique ;

5. seulement après, générer
   \[
   \mathcal C_\perp^{(u)},\quad \mathcal C_i^{(u)}.
   \]


In [9]:

artifact = {
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2",
    "final_status":FINAL_STATUS,
    "Q_shape":[10,10],
    "det_Gm":str(detGm),
    "det_Schur":str(detSchur),
    "det_Q_factored":str(detQ_factored),
    "generic_rank_witness":10,
    "generic_inverse_validated":True,
    "gates":GATES,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.1_Exact_GVH_Tensor_Coefficient_Lift_into_the_Coupled_Kinetic_Matrix.ipynb"
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path = export_dir/"gvh_0.3.2.7.3.7.2_coupled_kinetic_matrix.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:", artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2_coupled_kinetic_matrix.json



# Conclusion

Le verrou de `0.3.2.7.3.7.1` est attaqué directement.

Le notebook construit un Hessien cinétique couplé \(10\times10\), utilise le complément de Schur, vérifie une branche générique de rang maximal et construit une inverse bloc explicite.

Mais il reste une différence essentielle entre :

\[
\text{modèle cinétique générique paramétré}
\]

et

\[
\text{Hessien exact GVH full-field}.
\]

Le verdict est donc :

\[
\boxed{\text{PARTIAL PASS}}
\]

avec

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]

La prochaine sous-étape doit injecter les coefficients tensoriels GVH exacts dans \(Q_{AB}\).
